# Chume: OBSCURA final submission

Self-contained reproduction of the final Chume submission. The cell below writes the `chume` package (code only) to `/kaggle/working/chume`, installs it, builds chip features from `/kaggle/input/obscura`, trains the final configuration with site-grouped CV, and writes `submission.csv`.

Hardware used for the reference run: 4 CPU cores, 15 GB RAM, no GPU. Feature build about 2 minutes per 4,439 chips with 4 workers; training about 3 minutes per seed (3 seeds). No internet access and no external data or pretrained weights are needed for the final configuration.

In [ ]:
import json, os, pathlib
FILES = json.loads("{\"src/chume/__init__.py\": \"\", \"src/chume/ablate.py\": \"from __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom .config import load_config\\nfrom .train import run\\n\\nABLATIONS = [\\n    (\\n        \\\"01_tabular\\\",\\n        {\\n            \\\"features\\\": {\\\"generic_chip\\\": False, \\\"plume\\\": False, \\\"massbalance\\\": False},\\n            \\\"models\\\": {\\\"chain\\\": False, \\\"gate\\\": False, \\\"blend\\\": False, \\\"catboost\\\": False},\\n            \\\"targets\\\": {\\\"transform\\\": \\\"none\\\"},\\n        },\\n        \\\"true\\\",\\n    ),\\n    (\\n        \\\"02_generic_chip\\\",\\n        {\\n            \\\"features\\\": {\\\"plume\\\": False, \\\"massbalance\\\": False},\\n            \\\"models\\\": {\\\"chain\\\": False, \\\"gate\\\": False, \\\"blend\\\": False, \\\"catboost\\\": False},\\n            \\\"targets\\\": {\\\"transform\\\": \\\"none\\\"},\\n        },\\n        \\\"true\\\",\\n    ),\\n    (\\n        \\\"03_plume\\\",\\n        {\\n            \\\"features\\\": {\\\"massbalance\\\": False},\\n            \\\"models\\\": {\\\"chain\\\": False, \\\"gate\\\": False, \\\"blend\\\": False, \\\"catboost\\\": False},\\n            \\\"targets\\\": {\\\"transform\\\": \\\"none\\\"},\\n        },\\n        \\\"true\\\",\\n    ),\\n    (\\n        \\\"04_plume_random\\\",\\n        {\\n            \\\"features\\\": {\\\"massbalance\\\": False},\\n            \\\"models\\\": {\\\"chain\\\": False, \\\"gate\\\": False, \\\"blend\\\": False, \\\"catboost\\\": False},\\n            \\\"targets\\\": {\\\"transform\\\": \\\"none\\\"},\\n        },\\n        \\\"rand\\\",\\n    ),\\n    (\\n        \\\"05_massbalance\\\",\\n        {\\\"models\\\": {\\\"chain\\\": False, \\\"gate\\\": False, \\\"blend\\\": False, \\\"catboost\\\": False}},\\n        \\\"true\\\",\\n    ),\\n    (\\\"06_chain\\\", {\\\"models\\\": {\\\"gate\\\": False, \\\"blend\\\": False, \\\"catboost\\\": False}}, \\\"true\\\"),\\n    (\\\"07_gate\\\", {\\\"models\\\": {\\\"blend\\\": False, \\\"catboost\\\": False}}, \\\"true\\\"),\\n    (\\\"08_full_chume\\\", {}, \\\"true\\\"),\\n]\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--processed\\\", default=\\\"data/processed\\\")\\n    ap.add_argument(\\\"--only\\\", nargs=\\\"*\\\", default=None)\\n    ap.add_argument(\\\"--out\\\", default=\\\"docs/ABLATIONS.md\\\")\\n    a = ap.parse_args()\\n    base = load_config(a.config)\\n    rows = []\\n    for name, over, feat in ABLATIONS:\\n        if a.only and name not in a.only:\\n            continue\\n        scores, per = [], []\\n        for s in base[\\\"cv\\\"][\\\"seeds\\\"]:\\n            cfg = load_config(a.config, {**over, \\\"seed\\\": s})\\n            rep = run(cfg, str(Path(a.processed) / f\\\"features_{feat}.parquet\\\"), f\\\"abl_{name}_s{s}\\\")\\n            scores.append(rep[\\\"final\\\"][0])\\n            per.append(rep[\\\"final\\\"][1])\\n        targets = base[\\\"columns\\\"][\\\"targets\\\"]\\n        rows.append(\\n            (\\n                name,\\n                float(np.mean(scores)),\\n                float(np.std(scores)),\\n                {t: float(np.mean([p[t] for p in per])) for t in targets},\\n            )\\n        )\\n        print(name, rows[-1][1], rows[-1][2])\\n    targets = base[\\\"columns\\\"][\\\"targets\\\"]\\n    lines = [\\n        \\\"| Config | MCRMSE | std | \\\" + \\\" | \\\".join(targets) + \\\" |\\\",\\n        \\\"|\\\" + \\\"---|\\\" * (3 + len(targets)),\\n    ]\\n    for name, m, sd, pt in rows:\\n        lines.append(\\n            f\\\"| {name} | {m:.5f} | {sd:.5f} | \\\" + \\\" | \\\".join(f\\\"{pt[t]:.5f}\\\" for t in targets) + \\\" |\\\"\\n        )\\n    Path(a.out).parent.mkdir(parents=True, exist_ok=True)\\n    Path(a.out).write_text(\\\"# Ablations (repeated site-grouped CV)\\\\n\\\\n\\\" + \\\"\\\\n\\\".join(lines) + \\\"\\\\n\\\")\\n    Path(a.out).with_suffix(\\\".json\\\").write_text(json.dumps(rows, indent=2))\\n    print(\\\"\\\\n\\\".join(lines))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/adversarial.py\": \"from __future__ import annotations\\n\\nimport argparse\\nimport json\\n\\nimport lightgbm as lgb\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import roc_auc_score\\nfrom sklearn.model_selection import StratifiedKFold\\n\\nfrom .config import load_config\\nfrom .train import feature_columns\\n\\n\\ndef adversarial(df: pd.DataFrame, cols: list[str], seed: int = 42) -> tuple[float, pd.Series]:\\n    y = (df[\\\"_split\\\"] == \\\"test\\\").astype(int).values\\n    X = df[cols]\\n    oof = np.zeros(len(df))\\n    imp = np.zeros(len(cols))\\n    for tr, va in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):\\n        m = lgb.LGBMClassifier(\\n            n_estimators=400, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=seed\\n        )\\n        m.fit(X.iloc[tr], y[tr])\\n        oof[va] = m.predict_proba(X.iloc[va])[:, 1]\\n        imp += m.booster_.feature_importance(\\\"gain\\\")\\n    return float(roc_auc_score(y, oof)), pd.Series(imp, index=cols).sort_values(ascending=False)\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--features\\\", default=\\\"data/processed/features_true.parquet\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"outputs/adversarial.json\\\")\\n    a = ap.parse_args()\\n    cfg = load_config(a.config)\\n    df = pd.read_parquet(a.features)\\n    cols = feature_columns(df, cfg)\\n    auc, imp = adversarial(df, cols, cfg[\\\"seed\\\"])\\n    res = {\\\"auc\\\": auc, \\\"top_drift\\\": imp.head(25).to_dict()}\\n    with open(a.out, \\\"w\\\") as f:\\n        json.dump(res, f, indent=2)\\n    print(json.dumps(res, indent=2))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/chips.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\n\\nEPS = 1e-6\\n\\n\\nclass Chip:\\n    def __init__(\\n        self,\\n        arr: np.ndarray,\\n        band_order: list[str],\\n        scale: float = 10000.0,\\n        scl_bad=(0, 1, 3, 8, 9, 10),\\n        cldprb_bad: float = 40.0,\\n    ):\\n        arr = arr.astype(np.float32)\\n        self.bands: dict[str, np.ndarray] = {}\\n        for i, name in enumerate(band_order[: arr.shape[0]]):\\n            self.bands[name] = arr[i]\\n        self.scl = self.bands.pop(\\\"SCL\\\", None)\\n        self.cldprb = self.bands.pop(\\\"CLDPRB\\\", None)\\n        self.cldprb_bad = cldprb_bad\\n        for k in list(self.bands):\\n            b = self.bands[k]\\n            if scale and np.nanmax(b) > 2.0:\\n                b = b / scale\\n            self.bands[k] = b\\n        self.h, self.w = arr.shape[1], arr.shape[2]\\n        self.mask = self._mask(scl_bad)\\n\\n    def _mask(self, scl_bad) -> np.ndarray:\\n        m = np.ones((self.h, self.w), dtype=bool)\\n        for b in self.bands.values():\\n            m &= np.isfinite(b)\\n        if self.scl is not None:\\n            m &= ~np.isin(self.scl.astype(int), list(scl_bad))\\n            if self.cldprb is not None:\\n                m &= self.cldprb < self.cldprb_bad\\n        else:\\n            if \\\"B02\\\" in self.bands and \\\"B04\\\" in self.bands:\\n                bright = (self.bands[\\\"B02\\\"] + self.bands[\\\"B04\\\"]) / 2\\n                m &= bright < 0.35\\n            if \\\"B10\\\" in self.bands:\\n                m &= self.bands[\\\"B10\\\"] < 0.012\\n        return m\\n\\n    def get(self, name: str) -> np.ndarray | None:\\n        return self.bands.get(name)\\n\\n    def layers(self) -> dict[str, np.ndarray]:\\n        g = self.get\\n        out: dict[str, np.ndarray] = {}\\n        b11, b12, b8a, b08, b04, b01 = g(\\\"B11\\\"), g(\\\"B12\\\"), g(\\\"B8A\\\"), g(\\\"B08\\\"), g(\\\"B04\\\"), g(\\\"B01\\\")\\n        if b08 is None and b8a is not None:\\n            b08 = b8a\\n        if b11 is not None and b12 is not None:\\n            r = b12 / (b11 + EPS)\\n            out[\\\"r_ch4\\\"] = r\\n            med = np.nanmedian(r[self.mask]) if self.mask.any() else np.nanmedian(r)\\n            out[\\\"r_ch4_norm\\\"] = r / (med + EPS)\\n        if b12 is not None and b11 is not None and b8a is not None:\\n            out[\\\"tai\\\"] = (b12 - b11) / (b8a + EPS)\\n        if b12 is not None and b8a is not None:\\n            out[\\\"hot\\\"] = b12 - b8a\\n            out[\\\"hot_ratio\\\"] = b12 / (b8a + EPS)\\n        if b08 is not None and b04 is not None:\\n            out[\\\"ndvi\\\"] = (b08 - b04) / (b08 + b04 + EPS)\\n        if b11 is not None and b08 is not None:\\n            out[\\\"ndbi\\\"] = (b11 - b08) / (b11 + b08 + EPS)\\n        if b01 is not None and b04 is not None:\\n            out[\\\"haze\\\"] = b01 / (b04 + EPS)\\n        if b01 is not None:\\n            out[\\\"b01\\\"] = b01\\n        if b12 is not None:\\n            out[\\\"b12\\\"] = b12\\n        return out\\n\\n\\ndef generic_stats(chip: Chip) -> dict[str, float]:\\n    feats: dict[str, float] = {}\\n    m = chip.mask\\n    feats[\\\"clear_frac\\\"] = float(m.mean())\\n    for name, b in chip.bands.items():\\n        v = b[m] if m.any() else b.ravel()\\n        v = v[np.isfinite(v)]\\n        if v.size == 0:\\n            continue\\n        p10, p50, p90 = np.percentile(v, [10, 50, 90])\\n        feats[f\\\"g_{name}_p10\\\"], feats[f\\\"g_{name}_p50\\\"], feats[f\\\"g_{name}_p90\\\"] = (\\n            float(p10),\\n            float(p50),\\n            float(p90),\\n        )\\n    if chip.scl is not None:\\n        scl = chip.scl.astype(int)\\n        for c in range(12):\\n            feats[f\\\"scl_{c}\\\"] = float((scl == c).mean())\\n    return feats\\n\", \"src/chume/config.py\": \"from __future__ import annotations\\n\\nimport copy\\nimport random\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport yaml\\n\\n\\ndef deep_update(base: dict, upd: dict) -> dict:\\n    out = copy.deepcopy(base)\\n    for k, v in (upd or {}).items():\\n        if isinstance(v, dict) and isinstance(out.get(k), dict):\\n            out[k] = deep_update(out[k], v)\\n        else:\\n            out[k] = v\\n    return out\\n\\n\\ndef load_config(path: str | Path, overrides: dict | None = None) -> dict:\\n    path = Path(path)\\n    cfg = yaml.safe_load(path.read_text())\\n    if \\\"inherit\\\" in cfg:\\n        parent = load_config(path.parent / cfg.pop(\\\"inherit\\\"))\\n        cfg = deep_update(parent, cfg)\\n    if overrides:\\n        cfg = deep_update(cfg, overrides)\\n    return cfg\\n\\n\\ndef seed_everything(seed: int) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    try:\\n        import torch\\n\\n        torch.manual_seed(seed)\\n        torch.cuda.manual_seed_all(seed)\\n    except ImportError:\\n        pass\\n\", \"src/chume/cv.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold, StratifiedGroupKFold\\n\\n\\ndef availability_code(df: pd.DataFrame, targets: list[str]) -> np.ndarray:\\n    bits = np.stack([df[t].notna().values.astype(int) for t in targets], axis=1)\\n    return (bits * (2 ** np.arange(len(targets)))).sum(axis=1)\\n\\n\\ndef make_folds(\\n    df: pd.DataFrame, site_col: str, targets: list[str], n_folds: int = 5, seed: int = 42\\n) -> np.ndarray:\\n    groups = df[site_col].astype(str).values\\n    strat = availability_code(df, targets)\\n    counts = pd.Series(strat).value_counts()\\n    strat = np.where(pd.Series(strat).map(counts).values < n_folds, -1, strat)\\n    folds = np.full(len(df), -1, dtype=int)\\n    try:\\n        splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)\\n        splits = splitter.split(df, strat, groups)\\n    except ValueError:\\n        rng = np.random.RandomState(seed)\\n        uniq = np.unique(groups)\\n        perm = {g: i for i, g in enumerate(rng.permutation(uniq))}\\n        splits = GroupKFold(n_splits=n_folds).split(df, groups=[perm[g] for g in groups])\\n    for k, (_, va) in enumerate(splits):\\n        folds[va] = k\\n    assert (folds >= 0).all()\\n    return folds\\n\\n\\ndef check_group_integrity(folds: np.ndarray, groups: np.ndarray) -> None:\\n    g = pd.DataFrame({\\\"f\\\": folds, \\\"g\\\": groups})\\n    n = g.groupby(\\\"g\\\")[\\\"f\\\"].nunique()\\n    bad = n[n > 1]\\n    if len(bad):\\n        raise AssertionError(f\\\"{len(bad)} sites appear in more than one fold\\\")\\n\", \"src/chume/experiment.py\": \"\\\"\\\"\\\"Run one config override over the 3 CV seeds and append the result to docs/EXPERIMENTS.md.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom .config import load_config\\nfrom .train import run\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"name\\\")\\n    ap.add_argument(\\\"--over\\\", default=\\\"{}\\\", help=\\\"JSON config override (applied after --over_file)\\\")\\n    ap.add_argument(\\\"--over_file\\\", default=None, help=\\\"JSON file with a base override\\\")\\n    ap.add_argument(\\\"--features\\\", default=\\\"data/processed/features_true.parquet\\\")\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--seeds\\\", default=None, help=\\\"comma list; default cv.seeds\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"docs/EXPERIMENTS.md\\\")\\n    a = ap.parse_args()\\n    from .config import deep_update\\n\\n    over = json.loads(open(a.over_file).read()) if a.over_file else {}\\n    over = deep_update(over, json.loads(a.over))\\n    base = load_config(a.config, over)\\n    seeds = [int(s) for s in a.seeds.split(\\\",\\\")] if a.seeds else base[\\\"cv\\\"][\\\"seeds\\\"]\\n    targets = base[\\\"columns\\\"][\\\"targets\\\"]\\n    scores, per = [], []\\n    for s in seeds:\\n        rep = run(load_config(a.config, {**over, \\\"seed\\\": s}), a.features, f\\\"exp_{a.name}_s{s}\\\")\\n        scores.append(rep[\\\"final\\\"][0])\\n        per.append(rep[\\\"final\\\"][1])\\n        print(a.name, s, round(rep[\\\"final\\\"][0], 5), flush=True)\\n    m, sd = float(np.mean(scores)), float(np.std(scores))\\n    pt = {t: float(np.mean([p[t] for p in per])) for t in targets}\\n    out = Path(a.out)\\n    if not out.exists():\\n        out.write_text(\\n            \\\"# Experiments (site-grouped CV, mean over seeds)\\\\n\\\\n| name | MCRMSE | std | \\\"\\n            + \\\" | \\\".join(targets)\\n            + \\\" | seeds | override |\\\\n|\\\"\\n            + \\\"---|\\\" * (len(targets) + 5)\\n            + \\\"\\\\n\\\"\\n        )\\n    with out.open(\\\"a\\\") as f:\\n        f.write(\\n            f\\\"| {a.name} | {m:.5f} | {sd:.5f} | \\\"\\n            + \\\" | \\\".join(f\\\"{pt[t]:.4f}\\\" for t in targets)\\n            + f\\\" | {len(seeds)} | `{json.dumps(over)}` |\\\\n\\\"\\n        )\\n    print(f\\\"{a.name}: {m:.5f} +- {sd:.5f}\\\", json.dumps({t: round(v, 4) for t, v in pt.items()}))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/features/__init__.py\": \"\", \"src/chume/features/build.py\": \"from __future__ import annotations\\n\\nimport argparse\\nimport zlib\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom joblib import Parallel, delayed\\n\\nfrom ..chips import Chip, generic_stats\\nfrom ..config import load_config\\nfrom ..io import find_chip, load_chip, read_table\\nfrom .massbalance import add_massbalance, add_wind\\nfrom .plume import plume_features, wind_uv\\nfrom .site_context import add_site_context\\n\\n\\ndef _one(row: pd.Series, cfg: dict) -> dict:\\n    cols, cc, fc = cfg[\\\"columns\\\"], cfg[\\\"chips\\\"], cfg[\\\"features\\\"]\\n    sid = str(row[cols[\\\"id\\\"]])\\n    out: dict = {cols[\\\"id\\\"]: sid}\\n    path = find_chip(cfg[\\\"paths\\\"][\\\"chip_dir\\\"], sid, cc[\\\"pattern\\\"])\\n    if path is None:\\n        out[\\\"chip_missing\\\"] = 1.0\\n        return out\\n    arr = load_chip(path)\\n    chip = Chip(arr, cc[\\\"band_order\\\"], cc[\\\"scale\\\"], cc[\\\"scl_bad\\\"], cc.get(\\\"cldprb_bad\\\", 40.0))\\n    out[\\\"chip_missing\\\"] = 0.0\\n    if fc.get(\\\"generic_chip\\\", True):\\n        out.update(generic_stats(chip))\\n    if fc.get(\\\"plume\\\", True):\\n        u, v = wind_uv(row, cols)\\n        rng = np.random.RandomState(zlib.crc32(sid.encode()) % (2**31))\\n        out.update(plume_features(chip, u, v, cfg, rng))\\n    return out\\n\\n\\ndef build(cfg: dict, n_jobs: int = -1) -> pd.DataFrame:\\n    cols = cfg[\\\"columns\\\"]\\n    tr = read_table(cfg[\\\"paths\\\"][\\\"train_csv\\\"], cols[\\\"id\\\"])\\n    te = read_table(cfg[\\\"paths\\\"][\\\"test_csv\\\"], cols[\\\"id\\\"])\\n    tr[\\\"_split\\\"], te[\\\"_split\\\"] = \\\"train\\\", \\\"test\\\"\\n    df = pd.concat([tr, te], ignore_index=True)\\n    rows = Parallel(n_jobs=n_jobs, prefer=\\\"processes\\\")(delayed(_one)(r, cfg) for _, r in df.iterrows())\\n    chipf = pd.DataFrame(rows)\\n    df = df.merge(chipf, on=cols[\\\"id\\\"], how=\\\"left\\\")\\n    uv = np.array([wind_uv(r, cols) for _, r in df.iterrows()], dtype=float)\\n    df = add_wind(df, uv[:, 0], uv[:, 1])\\n    if cfg[\\\"features\\\"].get(\\\"massbalance\\\", True):\\n        df = add_massbalance(df, cfg[\\\"features\\\"][\\\"plume_radii\\\"])\\n    if cfg[\\\"features\\\"].get(\\\"site_context\\\", False):\\n        base = [c for c in df.columns if c.startswith((\\\"p_\\\", \\\"hot_\\\", \\\"tai_\\\", \\\"g_\\\"))]\\n        df = add_site_context(df, cols[\\\"site\\\"], base)\\n    return df\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--out\\\", default=None)\\n    ap.add_argument(\\\"--n_jobs\\\", type=int, default=-1)\\n    ap.add_argument(\\\"--random_angle\\\", action=\\\"store_true\\\")\\n    a = ap.parse_args()\\n    cfg = load_config(a.config, {\\\"features\\\": {\\\"plume_random_angle\\\": True}} if a.random_angle else None)\\n    df = build(cfg, a.n_jobs)\\n    tag = \\\"rand\\\" if cfg[\\\"features\\\"].get(\\\"plume_random_angle\\\") else \\\"true\\\"\\n    out = Path(a.out or Path(cfg[\\\"paths\\\"][\\\"processed\\\"]) / f\\\"features_{tag}.parquet\\\")\\n    out.parent.mkdir(parents=True, exist_ok=True)\\n    df.to_parquet(out, index=False)\\n    print(f\\\"wrote {out} shape={df.shape}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/features/embeddings.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\n\\nS2_13 = [\\\"B01\\\", \\\"B02\\\", \\\"B03\\\", \\\"B04\\\", \\\"B05\\\", \\\"B06\\\", \\\"B07\\\", \\\"B08\\\", \\\"B8A\\\", \\\"B09\\\", \\\"B10\\\", \\\"B11\\\", \\\"B12\\\"]\\n\\n\\nLOCAL_WEIGHTS = {\\n    \\\"ResNet50_Weights.SENTINEL2_ALL_MOCO\\\": \\\"data/weights/resnet50_sentinel2_all_moco-df8b932e.pth\\\",\\n}\\n\\n\\ndef load_encoder(weight_name: str = \\\"ResNet50_Weights.SENTINEL2_ALL_MOCO\\\"):\\n    import torch\\n\\n    try:\\n        from torchgeo.models import get_weight, resnet18, resnet50\\n    except ImportError:\\n        # torchgeo not installed: load the same published SSL4EO-S12 checkpoint\\n        # (downloaded from hf.co/torchgeo) into an identical timm ResNet50.\\n        import timm\\n\\n        model = timm.create_model(\\\"resnet50\\\", in_chans=13, num_classes=0)\\n        sd = torch.load(LOCAL_WEIGHTS[weight_name], map_location=\\\"cpu\\\")\\n        sd = {k: v for k, v in sd.items() if not k.startswith(\\\"fc.\\\")}\\n        missing, unexpected = model.load_state_dict(sd, strict=False)\\n        if missing or unexpected:\\n            raise RuntimeError(f\\\"weight mismatch: missing={missing} unexpected={unexpected}\\\")\\n        model.eval()\\n        return model, 13\\n\\n    w = get_weight(weight_name)\\n    arch = w.meta.get(\\\"model\\\", \\\"resnet50\\\")\\n    model = (resnet50 if arch == \\\"resnet50\\\" else resnet18)(weights=w)\\n    model.fc = torch.nn.Identity()\\n    model.eval()\\n    return model, w.meta.get(\\\"in_chans\\\", 13)\\n\\n\\ndef to_13(bands: dict[str, np.ndarray]) -> np.ndarray:\\n    h, w = next(iter(bands.values())).shape\\n    out = np.zeros((13, h, w), dtype=np.float32)\\n    for i, b in enumerate(S2_13):\\n        if b in bands:\\n            out[i] = bands[b]\\n        elif b == \\\"B10\\\":\\n            out[i] = 0.0\\n        elif b == \\\"B09\\\" and \\\"B8A\\\" in bands:\\n            out[i] = bands[\\\"B8A\\\"]\\n    return out\\n\\n\\ndef embed_batch(\\n    model, chips_13: list[np.ndarray], size: int = 224, center_crop: float | None = None\\n) -> np.ndarray:\\n    import torch\\n    import torch.nn.functional as F\\n\\n    xs = []\\n    for a in chips_13:\\n        x = torch.from_numpy(a)[None]\\n        if center_crop:\\n            h, w = x.shape[-2:]\\n            ch, cw = int(h * center_crop), int(w * center_crop)\\n            y0, x0 = (h - ch) // 2, (w - cw) // 2\\n            x = x[..., y0 : y0 + ch, x0 : x0 + cw]\\n        x = F.interpolate(x, size=(size, size), mode=\\\"bilinear\\\", align_corners=False)\\n        xs.append(x)\\n    with torch.no_grad():\\n        z = model(torch.cat(xs))\\n    return z.cpu().numpy()\\n\\n\\ndef main() -> None:\\n    import argparse\\n\\n    import pandas as pd\\n    from sklearn.decomposition import PCA\\n\\n    from ..chips import Chip\\n    from ..config import load_config\\n    from ..io import find_chip, load_chip\\n\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--features\\\", default=\\\"data/processed/features_true.parquet\\\")\\n    ap.add_argument(\\\"--batch\\\", type=int, default=32)\\n    a = ap.parse_args()\\n    cfg = load_config(a.config)\\n    c, cc, fc = cfg[\\\"columns\\\"], cfg[\\\"chips\\\"], cfg[\\\"features\\\"]\\n    df = pd.read_parquet(a.features)\\n    model, _ = load_encoder(fc[\\\"embeddings_model\\\"])\\n    Z_g, Z_c = [], []\\n    ids = df[c[\\\"id\\\"]].astype(str).tolist()\\n    for i in range(0, len(ids), a.batch):\\n        batch = []\\n        for sid in ids[i : i + a.batch]:\\n            p = find_chip(cfg[\\\"paths\\\"][\\\"chip_dir\\\"], sid, cc[\\\"pattern\\\"])\\n            chip = Chip(load_chip(p), cc[\\\"band_order\\\"], cc[\\\"scale\\\"], cc[\\\"scl_bad\\\"], cc.get(\\\"cldprb_bad\\\", 40.0))\\n            batch.append(to_13(chip.bands))\\n        Z_g.append(embed_batch(model, batch))\\n        Z_c.append(embed_batch(model, batch, center_crop=0.5))\\n    Z_g, Z_c = np.concatenate(Z_g), np.concatenate(Z_c)\\n    is_tr = (df[\\\"_split\\\"] == \\\"train\\\").values\\n    k = fc[\\\"embeddings_pca\\\"]\\n    for name, Z in ((\\\"g\\\", Z_g), (\\\"c\\\", Z_c)):\\n        pca = PCA(n_components=k, random_state=cfg[\\\"seed\\\"]).fit(Z[is_tr])\\n        P = pca.transform(Z)\\n        for j in range(k):\\n            df[f\\\"emb_{name}{j}\\\"] = P[:, j]\\n    df.to_parquet(a.features, index=False)\\n    print(f\\\"added {2*k} embedding columns to {a.features}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/features/massbalance.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef add_wind(df: pd.DataFrame, u: np.ndarray, v: np.ndarray) -> pd.DataFrame:\\n    df = df.copy()\\n    ws = np.hypot(u, v)\\n    df[\\\"mb_u\\\"], df[\\\"mb_v\\\"], df[\\\"mb_ws\\\"] = u, v, ws\\n    df[\\\"mb_inv_ws\\\"] = 1.0 / np.clip(ws, 0.5, None)\\n    df[\\\"mb_log_ws\\\"] = np.log(np.clip(ws, 0.1, None))\\n    return df\\n\\n\\ndef add_massbalance(df: pd.DataFrame, radii: list[float]) -> pd.DataFrame:\\n    df = df.copy()\\n    inv = df[\\\"mb_inv_ws\\\"]\\n    ws = df[\\\"mb_ws\\\"]\\n    cand = [\\n        c\\n        for c in df.columns\\n        if c.startswith(\\\"hot_n\\\") or c in (\\\"hot_ratio_max\\\", \\\"tai_max\\\", \\\"tai_n_pos_all\\\")\\n    ]\\n    cand += [\\n        c\\n        for c in df.columns\\n        if c.startswith(\\\"p_tai_core\\\") or c.startswith(\\\"p_hot_core\\\") or c.startswith(\\\"p_ndbi_core_mean\\\")\\n    ]\\n    for c in cand:\\n        df[f\\\"mb_{c}_per_ws\\\"] = df[c] * inv\\n    for rad in radii:\\n        for layer in (\\\"r_ch4_norm\\\", \\\"r_ch4\\\", \\\"haze\\\", \\\"b01\\\"):\\n            c = f\\\"p_{layer}_du{rad}\\\"\\n            if c in df.columns:\\n                df[f\\\"mb_{c}_x_ws\\\"] = df[c] * ws\\n    return df\\n\\n\\ndef target_forward(y: np.ndarray, kind: str, ws: np.ndarray) -> np.ndarray:\\n    if kind == \\\"none\\\":\\n        return y\\n    if kind == \\\"ws_mult\\\":\\n        return y * np.clip(ws, 0.5, None)\\n    if kind == \\\"slog\\\":\\n        return np.sign(y) * np.log1p(np.abs(y))\\n    raise ValueError(kind)\\n\\n\\ndef target_inverse(z: np.ndarray, kind: str, ws: np.ndarray) -> np.ndarray:\\n    if kind == \\\"none\\\":\\n        return z\\n    if kind == \\\"ws_mult\\\":\\n        return z / np.clip(ws, 0.5, None)\\n    if kind == \\\"slog\\\":\\n        return np.sign(z) * np.expm1(np.abs(z))\\n    raise ValueError(kind)\\n\", \"src/chume/features/plume.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom ..chips import Chip\\n\\n\\ndef wind_uv(row: pd.Series, cols: dict) -> tuple[float, float]:\\n    if cols.get(\\\"wind_u\\\") and cols.get(\\\"wind_v\\\"):\\n        return float(row[cols[\\\"wind_u\\\"]]), float(row[cols[\\\"wind_v\\\"]])\\n    if cols.get(\\\"wind_speed\\\") and cols.get(\\\"wind_dir\\\"):\\n        ws = float(row[cols[\\\"wind_speed\\\"]])\\n        d = np.deg2rad(float(row[cols[\\\"wind_dir\\\"]]))\\n        conv = cols.get(\\\"wind_dir_convention\\\", \\\"meteorological_from\\\")\\n        if conv == \\\"meteorological_from\\\":\\n            return -ws * np.sin(d), -ws * np.cos(d)\\n        if conv == \\\"meteorological_to\\\":\\n            return ws * np.sin(d), ws * np.cos(d)\\n        if conv == \\\"math_to\\\":\\n            return ws * np.cos(d), ws * np.sin(d)\\n        raise ValueError(conv)\\n    return float(\\\"nan\\\"), float(\\\"nan\\\")\\n\\n\\ndef polar_grid(h: int, w: int, north_up: bool = True) -> tuple[np.ndarray, np.ndarray]:\\n    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)\\n    cy, cx = (h - 1) / 2.0, (w - 1) / 2.0\\n    dx = xx - cx\\n    dy = (cy - yy) if north_up else (yy - cy)\\n    r = np.sqrt(dx**2 + dy**2) / (min(h, w) / 2.0)\\n    theta = np.arctan2(dy, dx)\\n    return r, theta\\n\\n\\ndef region_masks(\\n    h: int, w: int, wind_angle: float, radii: list[float], half_angle_deg: float, north_up: bool = True\\n):\\n    r, theta = polar_grid(h, w, north_up)\\n    rel = np.angle(np.exp(1j * (theta - wind_angle)))\\n    a = np.deg2rad(half_angle_deg)\\n    down = np.abs(rel) <= a\\n    up = np.abs(rel) >= np.pi - a\\n    cross = (np.abs(rel) >= np.pi / 2 - a) & (np.abs(rel) <= np.pi / 2 + a)\\n    out = {}\\n    r_core = radii[0]\\n    out[\\\"core\\\"] = r <= r_core\\n    for rad in radii:\\n        ring = (r > r_core) & (r <= rad) if rad > r_core else (r <= rad)\\n        out[f\\\"down_{rad}\\\"] = ring & down\\n        out[f\\\"up_{rad}\\\"] = ring & up\\n        out[f\\\"cross_{rad}\\\"] = ring & cross\\n        out[f\\\"ring_{rad}\\\"] = ring\\n    return out\\n\\n\\ndef _stat(v: np.ndarray) -> tuple[float, float, float]:\\n    v = v[np.isfinite(v)]\\n    if v.size == 0:\\n        return np.nan, np.nan, np.nan\\n    return float(v.mean()), float(np.percentile(v, 90)), float(v.max())\\n\\n\\ndef plume_features(\\n    chip: Chip, u: float, v: float, cfg: dict, rng: np.random.RandomState | None = None\\n) -> dict[str, float]:\\n    fc = cfg[\\\"features\\\"]\\n    radii = fc[\\\"plume_radii\\\"]\\n    ws = float(np.hypot(u, v)) if np.isfinite(u) and np.isfinite(v) else np.nan\\n    angle = float(np.arctan2(v, u)) if np.isfinite(ws) else 0.0\\n    if fc.get(\\\"plume_random_angle\\\"):\\n        rng = rng or np.random.RandomState(0)\\n        angle = float(rng.uniform(-np.pi, np.pi))\\n    calm = (not np.isfinite(ws)) or ws < fc[\\\"calm_ws\\\"]\\n    masks = region_masks(\\n        chip.h, chip.w, angle, radii, fc[\\\"plume_half_angle_deg\\\"], cfg[\\\"chips\\\"].get(\\\"north_up\\\", True)\\n    )\\n    clear = chip.mask\\n    feats: dict[str, float] = {\\\"is_calm\\\": float(calm)}\\n    layers = chip.layers()\\n    bg_std = {}\\n    for name, L in layers.items():\\n        bg = L[clear & ~masks[\\\"core\\\"]]\\n        bg = bg[np.isfinite(bg)]\\n        bg_std[name] = float(bg.std()) if bg.size > 10 else np.nan\\n    for name, L in layers.items():\\n        core_m, core_p90, core_max = _stat(L[masks[\\\"core\\\"] & clear])\\n        feats[f\\\"p_{name}_core_mean\\\"], feats[f\\\"p_{name}_core_p90\\\"], feats[f\\\"p_{name}_core_max\\\"] = (\\n            core_m,\\n            core_p90,\\n            core_max,\\n        )\\n        for rad in radii:\\n            st = {}\\n            for reg in (\\\"down\\\", \\\"up\\\", \\\"cross\\\", \\\"ring\\\"):\\n                st[reg] = _stat(L[masks[f\\\"{reg}_{rad}\\\"] & clear])\\n            for reg in (\\\"down\\\", \\\"up\\\", \\\"cross\\\"):\\n                feats[f\\\"p_{name}_{reg}{rad}_mean\\\"] = st[reg][0]\\n                feats[f\\\"p_{name}_{reg}{rad}_p90\\\"] = st[reg][1]\\n            d_minus_u = st[\\\"down\\\"][0] - st[\\\"up\\\"][0]\\n            feats[f\\\"p_{name}_du{rad}\\\"] = d_minus_u\\n            feats[f\\\"p_{name}_dc{rad}\\\"] = st[\\\"down\\\"][0] - st[\\\"cross\\\"][0]\\n            feats[f\\\"p_{name}_cr{rad}\\\"] = core_m - st[\\\"ring\\\"][0]\\n            s = bg_std[name]\\n            feats[f\\\"p_{name}_duz{rad}\\\"] = d_minus_u / s if s and np.isfinite(s) and s > 0 else np.nan\\n            if calm:\\n                feats[f\\\"p_{name}_du{rad}\\\"] = np.nan\\n                feats[f\\\"p_{name}_duz{rad}\\\"] = np.nan\\n    if \\\"hot_ratio\\\" in layers:\\n        hr = layers[\\\"hot_ratio\\\"]\\n        for thr in (1.2, 1.5, 2.0):\\n            feats[f\\\"hot_n{thr}_core\\\"] = float(((hr > thr) & masks[\\\"core\\\"]).sum())\\n            feats[f\\\"hot_n{thr}_all\\\"] = float((hr > thr).sum())\\n        feats[\\\"hot_ratio_max\\\"] = float(np.nanmax(hr))\\n    if \\\"tai\\\" in layers:\\n        tai = layers[\\\"tai\\\"]\\n        feats[\\\"tai_n_pos_all\\\"] = float((tai > 0.45).sum())\\n        feats[\\\"tai_max\\\"] = float(np.nanmax(tai))\\n    for rad in radii:\\n        feats[f\\\"clear_down{rad}\\\"] = (\\n            float(clear[masks[f\\\"down_{rad}\\\"]].mean()) if masks[f\\\"down_{rad}\\\"].any() else np.nan\\n        )\\n        feats[f\\\"clear_up{rad}\\\"] = (\\n            float(clear[masks[f\\\"up_{rad}\\\"]].mean()) if masks[f\\\"up_{rad}\\\"].any() else np.nan\\n        )\\n    feats[\\\"clear_core\\\"] = float(clear[masks[\\\"core\\\"]].mean()) if masks[\\\"core\\\"].any() else np.nan\\n    return feats\\n\", \"src/chume/features/site_context.py\": \"from __future__ import annotations\\n\\nimport pandas as pd\\n\\n\\ndef add_site_context(\\n    df: pd.DataFrame, site_col: str, cols: list[str], split_col: str = \\\"_split\\\"\\n) -> pd.DataFrame:\\n    df = df.copy()\\n    keys = [split_col, site_col]\\n    means = df.groupby(keys)[cols].transform(\\\"mean\\\")\\n    counts = df.groupby(keys)[cols[0]].transform(\\\"size\\\")\\n    for c in cols:\\n        df[f\\\"sc_{c}_mean\\\"] = means[c]\\n        df[f\\\"sc_{c}_dev\\\"] = df[c] - means[c]\\n    df[\\\"sc_n_rows\\\"] = counts\\n    return df\\n\", \"src/chume/figures.py\": \"from __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport matplotlib\\n\\nmatplotlib.use(\\\"Agg\\\")\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .features.plume import region_masks\\n\\n\\ndef ablation_bar(abl_json: str, out: Path) -> None:\\n    rows = json.loads(Path(abl_json).read_text())\\n    names = [r[0] for r in rows]\\n    m = [r[1] for r in rows]\\n    s = [r[2] for r in rows]\\n    fig, ax = plt.subplots(figsize=(9, 4))\\n    ax.barh(names, m, xerr=s, color=\\\"#4C78A8\\\")\\n    ax.invert_yaxis()\\n    ax.set_xlabel(\\\"MCRMSE (site-grouped CV, lower is better)\\\")\\n    lo = min(m) - 3 * max(s + [1e-6])\\n    ax.set_xlim(left=max(0, lo))\\n    fig.tight_layout()\\n    fig.savefig(out / \\\"ablation.png\\\", dpi=300)\\n\\n\\ndef region_diagram(out: Path) -> None:\\n    h = w = 128\\n    ms = region_masks(h, w, np.deg2rad(35), [0.10, 0.25, 0.45], 30)\\n    img = np.zeros((h, w, 3))\\n    img[ms[\\\"down_0.45\\\"]] = [0.85, 0.3, 0.2]\\n    img[ms[\\\"up_0.45\\\"]] = [0.2, 0.45, 0.85]\\n    img[ms[\\\"cross_0.45\\\"]] = [0.55, 0.55, 0.55]\\n    img[ms[\\\"core\\\"]] = [1.0, 0.85, 0.1]\\n    fig, ax = plt.subplots(figsize=(4.5, 4.5))\\n    ax.imshow(img)\\n    ax.annotate(\\n        \\\"\\\",\\n        xy=(64 + 40 * np.cos(np.deg2rad(35)), 64 - 40 * np.sin(np.deg2rad(35))),\\n        xytext=(64, 64),\\n        arrowprops=dict(arrowstyle=\\\"->\\\", color=\\\"k\\\", lw=2),\\n    )\\n    ax.set_title(\\\"Core (yellow), downwind (red),\\\\nupwind (blue), crosswind (grey)\\\", fontsize=10)\\n    ax.axis(\\\"off\\\")\\n    fig.tight_layout()\\n    fig.savefig(out / \\\"regions.png\\\", dpi=300)\\n\\n\\ndef oof_scatter(oof_csv: str, train_csv: str, id_col: str, targets: list[str], out: Path) -> None:\\n    o = pd.read_csv(oof_csv, dtype={id_col: str})\\n    t = pd.read_csv(train_csv, dtype={id_col: str})[[id_col] + targets]\\n    d = o.merge(t, on=id_col, suffixes=(\\\"_pred\\\", \\\"\\\"))\\n    fig, axes = plt.subplots(1, len(targets), figsize=(4 * len(targets), 4))\\n    for ax, tg in zip(axes, targets):\\n        m = d[tg].notna()\\n        ax.scatter(d.loc[m, tg], d.loc[m, f\\\"{tg}_pred\\\"], s=6, alpha=0.5)\\n        lo, hi = np.nanpercentile(d.loc[m, tg], [1, 99])\\n        ax.plot([lo, hi], [lo, hi], \\\"k--\\\", lw=1)\\n        ax.set_title(tg)\\n        ax.set_xlabel(\\\"actual\\\")\\n        ax.set_ylabel(\\\"OOF predicted\\\")\\n    fig.tight_layout()\\n    fig.savefig(out / \\\"oof_scatter.png\\\", dpi=300)\\n\\n\\ndef target_panels(train_csv: str, targets: list[str], out: Path) -> None:\\n    t = pd.read_csv(train_csv)[targets]\\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))\\n    c = t.corr()\\n    im = axes[0].imshow(c, cmap=\\\"RdBu_r\\\", vmin=-1, vmax=1)\\n    axes[0].set_xticks(range(len(targets)), targets, rotation=45)\\n    axes[0].set_yticks(range(len(targets)), targets)\\n    for i in range(len(targets)):\\n        for j in range(len(targets)):\\n            axes[0].text(j, i, f\\\"{c.iloc[i, j]:.2f}\\\", ha=\\\"center\\\", va=\\\"center\\\", fontsize=8)\\n    fig.colorbar(im, ax=axes[0])\\n    axes[0].set_title(\\\"Target correlation (co-observed)\\\")\\n    axes[1].bar(targets, t.isna().mean() * 100, color=\\\"#E45756\\\")\\n    axes[1].set_ylabel(\\\"% missing\\\")\\n    axes[1].set_title(\\\"Missing targets\\\")\\n    fig.tight_layout()\\n    fig.savefig(out / \\\"targets.png\\\", dpi=300)\\n\\n\\ndef importance_plot(imp_csv: str, out: Path, k: int = 20) -> None:\\n    imp = pd.read_csv(imp_csv, index_col=0)\\n    n = imp.shape[1]\\n    fig, axes = plt.subplots(1, n, figsize=(5 * n, 6))\\n    for ax, col in zip(np.atleast_1d(axes), imp.columns):\\n        s = imp[col].sort_values(ascending=False).head(k)[::-1]\\n        ax.barh(s.index, s.values, color=\\\"#54A24B\\\")\\n        ax.set_title(col)\\n        ax.tick_params(axis=\\\"y\\\", labelsize=7)\\n    fig.tight_layout()\\n    fig.savefig(out / \\\"importance.png\\\", dpi=200)\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--run\\\", required=True)\\n    ap.add_argument(\\\"--train\\\", default=\\\"data/raw/train.csv\\\")\\n    ap.add_argument(\\\"--ablations\\\", default=\\\"docs/ABLATIONS.json\\\")\\n    ap.add_argument(\\\"--id\\\", default=\\\"sample_id\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"outputs/figures\\\")\\n    a = ap.parse_args()\\n    targets = [\\\"t_ch4\\\", \\\"t_no2\\\", \\\"t_co\\\", \\\"t_aai\\\", \\\"t_firms\\\"]\\n    out = Path(a.out)\\n    out.mkdir(parents=True, exist_ok=True)\\n    region_diagram(out)\\n    target_panels(a.train, targets, out)\\n    oof_scatter(str(Path(a.run) / \\\"oof.csv\\\"), a.train, a.id, targets, out)\\n    importance_plot(str(Path(a.run) / \\\"importance_lgbm.csv\\\"), out)\\n    if Path(a.ablations).exists():\\n        ablation_bar(a.ablations, out)\\n    print(f\\\"figures in {out}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/io.py\": \"from __future__ import annotations\\n\\nimport glob\\nimport json\\nimport os\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef read_table(path: str | Path, id_col: str) -> pd.DataFrame:\\n    return pd.read_csv(path, dtype={id_col: str})\\n\\n\\ndef find_chip(chip_dir: str | Path, sample_id: str, pattern: str = \\\"{id}.*\\\") -> str | None:\\n    hits = glob.glob(os.path.join(str(chip_dir), \\\"**\\\", pattern.format(id=sample_id)), recursive=True)\\n    return sorted(hits)[0] if hits else None\\n\\n\\ndef load_chip(path: str) -> np.ndarray:\\n    ext = Path(path).suffix.lower()\\n    if ext == \\\".npy\\\":\\n        arr = np.load(path)\\n    elif ext == \\\".npz\\\":\\n        z = np.load(path)\\n        arr = z[list(z.keys())[0]]\\n    elif ext in (\\\".tif\\\", \\\".tiff\\\"):\\n        try:\\n            import warnings\\n\\n            import rasterio\\n\\n            with warnings.catch_warnings():\\n                warnings.simplefilter(\\\"ignore\\\", rasterio.errors.NotGeoreferencedWarning)\\n                with rasterio.open(path) as src:\\n                    arr = src.read()\\n        except ImportError:\\n            import tifffile\\n\\n            arr = tifffile.imread(path)\\n    elif ext in (\\\".png\\\", \\\".jpg\\\", \\\".jpeg\\\"):\\n        from PIL import Image\\n\\n        arr = np.asarray(Image.open(path))\\n    else:\\n        raise ValueError(f\\\"unknown chip format {ext}\\\")\\n    arr = np.asarray(arr)\\n    if arr.ndim == 2:\\n        arr = arr[None]\\n    if arr.ndim == 3 and arr.shape[-1] < arr.shape[0] and arr.shape[-1] <= 20:\\n        arr = np.transpose(arr, (2, 0, 1))\\n    return arr\\n\\n\\ndef run_dir(root: str | Path, run_id: str) -> Path:\\n    p = Path(root) / run_id\\n    p.mkdir(parents=True, exist_ok=True)\\n    return p\\n\\n\\ndef append_jsonl(path: str | Path, rec: dict) -> None:\\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\\n    with open(path, \\\"a\\\") as f:\\n        f.write(json.dumps(rec, default=float) + \\\"\\\\n\\\")\\n\", \"src/chume/metric.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef rmse_masked(y_true, y_pred) -> float:\\n    y_true = np.asarray(y_true, dtype=float)\\n    y_pred = np.asarray(y_pred, dtype=float)\\n    m = ~np.isnan(y_true)\\n    if m.sum() == 0:\\n        return float(\\\"nan\\\")\\n    return float(np.sqrt(np.mean((y_true[m] - y_pred[m]) ** 2)))\\n\\n\\ndef mcrmse(y_true: pd.DataFrame, y_pred: pd.DataFrame, targets: list[str]) -> tuple[float, dict]:\\n    scores = {t: rmse_masked(y_true[t].values, y_pred[t].values) for t in targets}\\n    valid = [v for v in scores.values() if not np.isnan(v)]\\n    return float(np.mean(valid)), scores\\n\", \"src/chume/models/__init__.py\": \"\", \"src/chume/models/blend.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nfrom scipy.optimize import nnls\\n\\n\\ndef fit_blend(y: np.ndarray, preds: np.ndarray) -> np.ndarray:\\n    m = ~np.isnan(y) & ~np.isnan(preds).any(axis=1)\\n    w, _ = nnls(preds[m], y[m])\\n    if w.sum() <= 0:\\n        w = np.ones(preds.shape[1])\\n    return w / w.sum()\\n\\n\\ndef apply_blend(preds: np.ndarray, w: np.ndarray) -> np.ndarray:\\n    return preds @ w\\n\\n\\ndef nested_blend_oof(\\n    y: np.ndarray, preds: np.ndarray, folds: np.ndarray\\n) -> tuple[np.ndarray, list[np.ndarray]]:\\n    out = np.full(len(y), np.nan)\\n    ws = []\\n    for k in range(int(folds.max()) + 1):\\n        tr = folds != k\\n        w = fit_blend(np.where(tr, y, np.nan), preds)\\n        ws.append(w)\\n        va = folds == k\\n        out[va] = apply_blend(preds[va], w)\\n    return out, ws\\n\", \"src/chume/models/chain.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .gbdt import FITTERS, fit_oof\\n\\n\\ndef _relabel(f: np.ndarray) -> np.ndarray:\\n    u = np.unique(f)\\n    m = {v: i for i, v in enumerate(u)}\\n    return np.array([m[v] for v in f])\\n\\n\\ndef strict_chain(\\n    X: pd.DataFrame,\\n    Y: pd.DataFrame,\\n    folds: np.ndarray,\\n    X_test: pd.DataFrame,\\n    params: dict,\\n    seed: int,\\n    include_self: bool = False,\\n    kind: str = \\\"lgbm\\\",\\n) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    targets = list(Y.columns)\\n    K = int(folds.max()) + 1\\n    oof = np.full((len(X), len(targets)), np.nan)\\n    test = np.zeros((len(X_test), len(targets)))\\n    fit = FITTERS[kind]\\n    for k in range(K):\\n        tr = folds != k\\n        va = folds == k\\n        f_in = _relabel(folds[tr])\\n        Xtr, Xva = X[tr].reset_index(drop=True), X[va].reset_index(drop=True)\\n        F_tr = pd.DataFrame(index=Xtr.index)\\n        F_va = pd.DataFrame(index=Xva.index)\\n        F_te = pd.DataFrame(index=X_test.index)\\n        for t in targets:\\n            ytr = Y[t].values[tr]\\n            o, te_p, ms = fit_oof(\\n                Xtr, ytr, f_in, pd.concat([Xva, X_test], ignore_index=True), kind, params, [seed]\\n            )\\n            mu = np.nanmean(ytr)\\n            F_tr[f\\\"s1_{t}\\\"] = np.where(np.isnan(o), mu, o)\\n            F_va[f\\\"s1_{t}\\\"] = te_p[: len(Xva)]\\n            F_te[f\\\"s1_{t}\\\"] = te_p[len(Xva) :]\\n        es_fold = 0\\n        for j, t in enumerate(targets):\\n            use = [c for c in F_tr.columns if include_self or c != f\\\"s1_{t}\\\"]\\n            A_tr = pd.concat([Xtr, F_tr[use]], axis=1)\\n            A_va = pd.concat([Xva, F_va[use]], axis=1)\\n            A_te = pd.concat([X_test.reset_index(drop=True), F_te[use].reset_index(drop=True)], axis=1)\\n            y = Y[t].values[tr]\\n            obs = ~np.isnan(y)\\n            es = (f_in == es_fold) & obs\\n            fit_m = (f_in != es_fold) & obs\\n            if es.sum() == 0 or fit_m.sum() < 20:\\n                continue\\n            m = fit(A_tr[fit_m], y[fit_m], A_tr[es], y[es], params, seed)\\n            oof[va, j] = m.predict(A_va)\\n            test[:, j] += m.predict(A_te) / K\\n    return pd.DataFrame(oof, columns=targets), pd.DataFrame(test, columns=targets)\\n\", \"src/chume/models/gate.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\n\\n\\ndef fit_gate(y: np.ndarray, pred: np.ndarray, mu: float, w_max: float = 1.2) -> float:\\n    m = ~np.isnan(y) & ~np.isnan(pred)\\n    d = pred[m] - mu\\n    r = y[m] - mu\\n    denom = float((d * d).sum())\\n    if denom <= 0:\\n        return 0.0\\n    return float(np.clip((d * r).sum() / denom, 0.0, w_max))\\n\\n\\ndef apply_gate(pred: np.ndarray, mu: float, w: float) -> np.ndarray:\\n    return mu + w * (pred - mu)\\n\\n\\ndef gate_stability(y: np.ndarray, pred: np.ndarray, folds: np.ndarray, mu: float) -> list[float]:\\n    return [fit_gate(np.where(folds == k, y, np.nan), pred, mu) for k in range(int(folds.max()) + 1)]\\n\\n\\ndef nested_gate_oof(\\n    y: np.ndarray, pred: np.ndarray, folds: np.ndarray, w_max: float = 1.2\\n) -> tuple[np.ndarray, list[float]]:\\n    out = np.full_like(pred, np.nan, dtype=float)\\n    ws = []\\n    for k in range(int(folds.max()) + 1):\\n        tr = (folds != k) & ~np.isnan(y)\\n        mu = float(np.nanmean(y[tr]))\\n        w = fit_gate(np.where(tr, y, np.nan), pred, mu, w_max)\\n        ws.append(w)\\n        va = folds == k\\n        out[va] = apply_gate(pred[va], mu, w)\\n    return out, ws\\n\", \"src/chume/models/gbdt.py\": \"from __future__ import annotations\\n\\nimport warnings\\n\\nwarnings.filterwarnings(\\\"ignore\\\", category=DeprecationWarning)\\nwarnings.filterwarnings(\\\"ignore\\\", message=\\\".*eval_set.*\\\")\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef _fit_lgbm(Xtr, ytr, Xva, yva, params, seed):\\n    import lightgbm as lgb\\n\\n    p = dict(params)\\n    n_est = p.pop(\\\"n_estimators\\\", 3000)\\n    es = p.pop(\\\"early_stopping_rounds\\\", 150)\\n    p[\\\"seed\\\"] = seed\\n    m = lgb.LGBMRegressor(n_estimators=n_est, **p)\\n    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(es, verbose=False)])\\n    return m\\n\\n\\ndef _fit_cat(Xtr, ytr, Xva, yva, params, seed):\\n    from catboost import CatBoostRegressor\\n\\n    p = dict(params)\\n    p[\\\"random_seed\\\"] = seed\\n    m = CatBoostRegressor(**p)\\n    m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)\\n    return m\\n\\n\\nFITTERS = {\\\"lgbm\\\": _fit_lgbm, \\\"catboost\\\": _fit_cat}\\n\\n\\ndef fit_oof(\\n    X: pd.DataFrame,\\n    y: np.ndarray,\\n    folds: np.ndarray,\\n    X_test: pd.DataFrame,\\n    kind: str,\\n    params: dict,\\n    seeds: list[int],\\n) -> tuple[np.ndarray, np.ndarray, list]:\\n    obs = ~np.isnan(y)\\n    K = int(folds.max()) + 1\\n    oof = np.full(len(y), np.nan)\\n    test_pred = np.zeros(len(X_test))\\n    models = []\\n    fit = FITTERS[kind]\\n    for k in range(K):\\n        va = (folds == k) & obs\\n        inner = (folds == (k + 1) % K) & obs\\n        tr = (folds != k) & (folds != (k + 1) % K) & obs\\n        if va.sum() == 0 or tr.sum() < 20 or inner.sum() == 0:\\n            continue\\n        pv = np.zeros(va.sum())\\n        for s in seeds:\\n            m = fit(X[tr], y[tr], X[inner], y[inner], params, s)\\n            pv += m.predict(X[va]) / len(seeds)\\n            test_pred += m.predict(X_test) / (len(seeds) * K)\\n            models.append(m)\\n        oof[va] = pv\\n    return oof, test_pred, models\\n\\n\\ndef importance(models: list, cols: list[str]) -> pd.Series:\\n    imp = np.zeros(len(cols))\\n    for m in models:\\n        if hasattr(m, \\\"feature_importances_\\\"):\\n            v = np.asarray(m.feature_importances_, dtype=float)\\n            imp += v / (v.sum() + 1e-12)\\n    return pd.Series(imp / max(len(models), 1), index=cols).sort_values(ascending=False)\\n\", \"src/chume/recon.py\": \"from __future__ import annotations\\n\\nimport argparse\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .config import load_config\\nfrom .io import find_chip, load_chip, read_table\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"docs/DATA_NOTES_auto.md\\\")\\n    ap.add_argument(\\\"--n_chips\\\", type=int, default=20)\\n    a = ap.parse_args()\\n    cfg = load_config(a.config)\\n    c = cfg[\\\"columns\\\"]\\n    tr = read_table(cfg[\\\"paths\\\"][\\\"train_csv\\\"], c[\\\"id\\\"])\\n    te = read_table(cfg[\\\"paths\\\"][\\\"test_csv\\\"], c[\\\"id\\\"])\\n    L = [\\\"# Auto recon\\\", \\\"\\\"]\\n    raw = Path(cfg[\\\"paths\\\"][\\\"raw\\\"])\\n    L += [\\\"## Files\\\", \\\"\\\"] + [\\n        f\\\"- {p.relative_to(raw)}  ({p.stat().st_size/1e6:.1f} MB)\\\"\\n        for p in sorted(raw.glob(\\\"*\\\"))\\n        if p.is_file()\\n    ]\\n    L += [\\n        \\\"\\\",\\n        f\\\"train {tr.shape}, test {te.shape}\\\",\\n        \\\"\\\",\\n        \\\"## Columns\\\",\\n        \\\"\\\",\\n        \\\"| col | dtype | train_na% | test_na% | train min/mean/max | test mean |\\\",\\n        \\\"|---|---|---|---|---|---|\\\",\\n    ]\\n    for col in tr.columns:\\n        tn = tr[col].isna().mean() * 100\\n        ten = te[col].isna().mean() * 100 if col in te else float(\\\"nan\\\")\\n        if pd.api.types.is_numeric_dtype(tr[col]):\\n            s = f\\\"{tr[col].min():.4g}/{tr[col].mean():.4g}/{tr[col].max():.4g}\\\"\\n            tm = f\\\"{te[col].mean():.4g}\\\" if col in te and pd.api.types.is_numeric_dtype(te[col]) else \\\"-\\\"\\n        else:\\n            s, tm = f\\\"{tr[col].nunique()} unique\\\", \\\"-\\\"\\n        L.append(f\\\"| {col} | {tr[col].dtype} | {tn:.1f} | {ten:.1f} | {s} | {tm} |\\\")\\n    T = [t for t in c[\\\"targets\\\"] if t in tr]\\n    if T:\\n        L += [\\n            \\\"\\\",\\n            \\\"## Targets\\\",\\n            \\\"\\\",\\n            tr[T].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.to_markdown(),\\n            \\\"\\\",\\n        ]\\n        L += [\\\"Skew:\\\", \\\"\\\", tr[T].skew().to_frame(\\\"skew\\\").to_markdown(), \\\"\\\"]\\n        L += [\\\"Pairwise corr (co-observed rows):\\\", \\\"\\\", tr[T].corr().round(3).to_markdown(), \\\"\\\"]\\n        pat = tr[T].notna().astype(int).astype(str).agg(\\\"\\\".join, axis=1).value_counts()\\n        L += [\\n            \\\"Availability patterns (order \\\" + \\\",\\\".join(T) + \\\"):\\\",\\n            \\\"\\\",\\n            pat.to_frame(\\\"n\\\").to_markdown(),\\n            \\\"\\\",\\n        ]\\n    if c[\\\"site\\\"] in tr:\\n        L += [\\n            f\\\"Sites: train {tr[c['site']].nunique()}, \\\"\\n            f\\\"test {te[c['site']].nunique() if c['site'] in te else 'n/a'}\\\",\\n            f\\\"Overlap: {len(set(tr[c['site']]) & set(te.get(c['site'], pd.Series(dtype=str))))}\\\",\\n            f\\\"Rows per site (train): {tr[c['site']].value_counts().describe().round(2).to_dict()}\\\",\\n            \\\"\\\",\\n        ]\\n    L += [\\\"## Chips\\\", \\\"\\\"]\\n    for sid in tr[c[\\\"id\\\"]].head(5):\\n        p = find_chip(cfg[\\\"paths\\\"][\\\"chip_dir\\\"], sid, cfg[\\\"chips\\\"][\\\"pattern\\\"])\\n        if p is None:\\n            L.append(f\\\"- {sid}: NOT FOUND\\\")\\n            continue\\n        arr = load_chip(p)\\n        L.append(\\n            f\\\"- {sid}: {p} shape={arr.shape} dtype={arr.dtype} \\\"\\n            f\\\"per-band max={np.nanmax(arr.reshape(arr.shape[0], -1), axis=1).round(1).tolist()}\\\"\\n        )\\n    try:\\n        import matplotlib\\n\\n        matplotlib.use(\\\"Agg\\\")\\n        import matplotlib.pyplot as plt\\n\\n        n = a.n_chips\\n        fig, axes = plt.subplots(4, n // 4 + (n % 4 > 0), figsize=(16, 13))\\n        bo = cfg[\\\"chips\\\"][\\\"band_order\\\"]\\n        for ax, sid in zip(axes.ravel(), tr[c[\\\"id\\\"]].sample(n, random_state=0)):\\n            p = find_chip(cfg[\\\"paths\\\"][\\\"chip_dir\\\"], sid, cfg[\\\"chips\\\"][\\\"pattern\\\"])\\n            ax.axis(\\\"off\\\")\\n            if p is None:\\n                continue\\n            arr = load_chip(p).astype(float)\\n            idx = [bo.index(b) for b in (\\\"B12\\\", \\\"B11\\\", \\\"B04\\\") if b in bo and bo.index(b) < arr.shape[0]]\\n            if len(idx) == 3:\\n                rgb = np.stack([arr[i] for i in idx], -1)\\n                rgb = np.clip(rgb / np.nanpercentile(rgb, 99), 0, 1)\\n                ax.imshow(rgb)\\n            ax.plot(arr.shape[2] / 2, arr.shape[1] / 2, \\\"c+\\\", ms=10)\\n        Path(\\\"outputs/recon\\\").mkdir(parents=True, exist_ok=True)\\n        fig.savefig(\\\"outputs/recon/contact_sheet_swir.png\\\", dpi=110, bbox_inches=\\\"tight\\\")\\n        L += [\\n            \\\"\\\",\\n            \\\"Contact sheet (B12/B11/B04 false color, cyan + at center): \\\"\\n            \\\"outputs/recon/contact_sheet_swir.png\\\",\\n        ]\\n    except Exception as e:\\n        L.append(f\\\"contact sheet failed: {e}\\\")\\n    Path(a.out).parent.mkdir(parents=True, exist_ok=True)\\n    Path(a.out).write_text(\\\"\\\\n\\\".join(L) + \\\"\\\\n\\\")\\n    print(f\\\"wrote {a.out}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/train.py\": \"from __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport subprocess\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .config import load_config, seed_everything\\nfrom .cv import check_group_integrity, make_folds\\nfrom .features.massbalance import target_forward, target_inverse\\nfrom .io import append_jsonl, run_dir\\nfrom .metric import mcrmse, rmse_masked\\nfrom .models.blend import apply_blend, fit_blend, nested_blend_oof\\nfrom .models.chain import strict_chain\\nfrom .models.gate import apply_gate, fit_gate, nested_gate_oof\\nfrom .models.gbdt import fit_oof, importance\\n\\nNON_FEATURE_PREFIX = (\\\"_\\\",)\\n\\n\\ndef feature_columns(df: pd.DataFrame, cfg: dict) -> list[str]:\\n    import re\\n\\n    c = cfg[\\\"columns\\\"]\\n    skip = {c[\\\"id\\\"], c[\\\"site\\\"], *c[\\\"targets\\\"], *c.get(\\\"drop\\\", [])}\\n    drop_re = [re.compile(r) for r in cfg[\\\"features\\\"].get(\\\"drop_regex\\\", []) or []]\\n    keep_re = [re.compile(r) for r in cfg[\\\"features\\\"].get(\\\"keep_regex\\\", []) or []]\\n    cols = []\\n    for col in df.columns:\\n        if col in skip or col.startswith(NON_FEATURE_PREFIX):\\n            continue\\n        if any(r.search(col) for r in drop_re) and not any(r.search(col) for r in keep_re):\\n            continue\\n        if not pd.api.types.is_numeric_dtype(df[col]):\\n            continue\\n        if not cfg[\\\"features\\\"].get(\\\"site_context\\\") and col.startswith(\\\"sc_\\\"):\\n            continue\\n        if not cfg[\\\"features\\\"].get(\\\"plume\\\", True) and (\\n            col.startswith((\\\"p_\\\", \\\"hot_\\\", \\\"tai_\\\", \\\"clear_\\\", \\\"is_calm\\\")) or col.startswith(\\\"mb_p_\\\")\\n        ):\\n            continue\\n        if (\\n            not cfg[\\\"features\\\"].get(\\\"massbalance\\\", True)\\n            and col.startswith(\\\"mb_\\\")\\n            and col not in (\\\"mb_u\\\", \\\"mb_v\\\", \\\"mb_ws\\\")\\n        ):\\n            continue\\n        if not cfg[\\\"features\\\"].get(\\\"generic_chip\\\", True) and (\\n            col.startswith((\\\"g_\\\", \\\"scl_\\\")) or col == \\\"clear_frac\\\"\\n        ):\\n            continue\\n        if not cfg[\\\"features\\\"].get(\\\"embeddings\\\", False) and col.startswith(\\\"emb_\\\"):\\n            continue\\n        if df[col].nunique(dropna=True) <= 1:\\n            continue\\n        cols.append(col)\\n    return cols\\n\\n\\ndef git_hash() -> str:\\n    try:\\n        return subprocess.check_output(\\n            [\\\"git\\\", \\\"rev-parse\\\", \\\"--short\\\", \\\"HEAD\\\"], text=True, stderr=subprocess.DEVNULL\\n        ).strip()\\n    except Exception:\\n        return \\\"nogit\\\"\\n\\n\\ndef pick_transform(X, y, ws, folds, Xte, cfg, seed) -> str:\\n    mode = cfg[\\\"targets\\\"][\\\"transform\\\"]\\n    if mode != \\\"auto\\\":\\n        return mode\\n    best, best_s = \\\"none\\\", np.inf\\n    obs = ~np.isnan(y)\\n    for kind in cfg[\\\"targets\\\"][\\\"candidates\\\"]:\\n        if kind == \\\"slog\\\" and np.nanmin(np.abs(y)) == np.inf:\\n            continue\\n        z = np.where(obs, target_forward(np.nan_to_num(y), kind, ws), np.nan)\\n        o, _, _ = fit_oof(X, z, folds, Xte.iloc[:1], \\\"lgbm\\\", cfg[\\\"lgbm_params\\\"], [seed])\\n        p = target_inverse(o, kind, ws)\\n        s = rmse_masked(y, p)\\n        if s < best_s:\\n            best, best_s = kind, s\\n    return best\\n\\n\\ndef run(cfg: dict, features_path: str, run_id: str) -> dict:\\n    t0 = time.time()\\n    seed = cfg[\\\"seed\\\"]\\n    seed_everything(seed)\\n    c = cfg[\\\"columns\\\"]\\n    targets = c[\\\"targets\\\"]\\n    df = pd.read_parquet(features_path)\\n    df[c[\\\"id\\\"]] = df[c[\\\"id\\\"]].astype(str)\\n    tr = df[df[\\\"_split\\\"] == \\\"train\\\"].reset_index(drop=True)\\n    te = df[df[\\\"_split\\\"] == \\\"test\\\"].reset_index(drop=True)\\n    cols = feature_columns(df, cfg)\\n    X, Xte = tr[cols], te[cols]\\n    Y = tr[targets]\\n    folds = make_folds(tr, c[\\\"site\\\"], targets, cfg[\\\"cv\\\"][\\\"n_folds\\\"], seed)\\n    check_group_integrity(folds, tr[c[\\\"site\\\"]].astype(str).values)\\n    out = run_dir(cfg[\\\"paths\\\"][\\\"runs\\\"], run_id)\\n    pd.DataFrame({c[\\\"id\\\"]: tr[c[\\\"id\\\"]], \\\"fold\\\": folds}).to_csv(out / \\\"folds.csv\\\", index=False)\\n    ws_tr = tr[\\\"mb_ws\\\"].fillna(tr[\\\"mb_ws\\\"].median()).values\\n    ws_te = te[\\\"mb_ws\\\"].fillna(tr[\\\"mb_ws\\\"].median()).values\\n    mseeds = [seed + i for i in range(cfg[\\\"models\\\"][\\\"n_model_seeds\\\"])]\\n\\n    stages: dict[str, dict[str, tuple[np.ndarray, np.ndarray]]] = {}\\n    transforms, imps = {}, {}\\n    for t in targets:\\n        y = Y[t].values.astype(float)\\n        kind = pick_transform(X, y, ws_tr, folds, Xte, cfg, seed)\\n        transforms[t] = kind\\n        obs = ~np.isnan(y)\\n        z = np.where(obs, target_forward(np.nan_to_num(y), kind, ws_tr), np.nan)\\n        for mk in (\\\"lgbm\\\", \\\"catboost\\\"):\\n            if not cfg[\\\"models\\\"].get(mk, True):\\n                continue\\n            params = cfg[\\\"lgbm_params\\\"] if mk == \\\"lgbm\\\" else cfg[\\\"catboost_params\\\"]\\n            o, p, ms = fit_oof(X, z, folds, Xte, mk, params, mseeds)\\n            stages.setdefault(mk, {})[t] = (\\n                target_inverse(o, kind, ws_tr),\\n                target_inverse(p, kind, ws_te),\\n            )\\n            if mk == \\\"lgbm\\\":\\n                imps[t] = importance(ms, cols)\\n\\n    if cfg[\\\"models\\\"].get(\\\"chain\\\", True):\\n        co, ct = strict_chain(\\n            X, Y, folds, Xte, cfg[\\\"lgbm_params\\\"], seed, cfg[\\\"models\\\"].get(\\\"chain_include_self\\\", False)\\n        )\\n        stages[\\\"chain\\\"] = {t: (co[t].values, ct[t].values) for t in targets}\\n\\n    final_oof = pd.DataFrame(index=tr.index)\\n    final_test = pd.DataFrame(index=te.index)\\n    report: dict = {\\\"stages\\\": {}, \\\"blend_w\\\": {}, \\\"gate_w\\\": {}, \\\"transforms\\\": transforms}\\n    for name, d in stages.items():\\n        o = pd.DataFrame({t: d[t][0] for t in targets})\\n        report[\\\"stages\\\"][name] = mcrmse(Y, o, targets)\\n    mean_o = pd.DataFrame({t: np.full(len(tr), np.nanmean(Y[t])) for t in targets})\\n    report[\\\"stages\\\"][\\\"mean\\\"] = mcrmse(Y, mean_o, targets)\\n\\n    names = list(stages)\\n    for t in targets:\\n        y = Y[t].values.astype(float)\\n        P = np.column_stack([stages[n][t][0] for n in names])\\n        Pt = np.column_stack([stages[n][t][1] for n in names])\\n        P = np.where(np.isnan(P), np.nanmean(y), P)\\n        if cfg[\\\"models\\\"].get(\\\"blend\\\", True) and len(names) > 1:\\n            bo, _ = nested_blend_oof(y, P, folds)\\n            w = fit_blend(y, P)\\n            bt = apply_blend(Pt, w)\\n        else:\\n            j0 = names.index(\\\"chain\\\") if \\\"chain\\\" in names else 0\\n            bo, bt, w = P[:, j0], Pt[:, j0], np.eye(len(names))[j0]\\n        report[\\\"blend_w\\\"][t] = dict(zip(names, map(float, w)))\\n        if cfg[\\\"models\\\"].get(\\\"gate\\\", True):\\n            w_max = float(cfg[\\\"models\\\"].get(\\\"gate_w_max\\\", 1.2))\\n            go, gws = nested_gate_oof(y, bo, folds, w_max)\\n            mu = float(np.nanmean(y))\\n            gw = fit_gate(y, bo, mu, w_max)\\n            gt = apply_gate(bt, mu, gw)\\n            report[\\\"gate_w\\\"][t] = {\\\"full\\\": gw, \\\"per_fold\\\": gws}\\n        else:\\n            go, gt = bo, bt\\n        final_oof[t], final_test[t] = go, gt\\n\\n    if cfg.get(\\\"post\\\", {}).get(\\\"clip_train_range\\\", False):\\n        for t in targets:\\n            lo, hi = float(np.nanmin(Y[t])), float(np.nanmax(Y[t]))\\n            final_oof[t] = np.clip(final_oof[t].values, lo, hi)\\n            final_test[t] = np.clip(final_test[t].values, lo, hi)\\n    report[\\\"final\\\"] = mcrmse(Y, final_oof, targets)\\n    sub = pd.DataFrame({c[\\\"id\\\"]: te[c[\\\"id\\\"]].astype(str)})\\n    for t in targets:\\n        v = final_test[t].values\\n        sub[t] = np.where(np.isfinite(v), v, np.nanmean(Y[t]))\\n    sub.to_csv(out / \\\"submission.csv\\\", index=False)\\n    oof_df = final_oof.copy()\\n    oof_df.insert(0, c[\\\"id\\\"], tr[c[\\\"id\\\"]])\\n    oof_df.to_csv(out / \\\"oof.csv\\\", index=False)\\n    pd.DataFrame(imps).to_csv(out / \\\"importance_lgbm.csv\\\")\\n    (out / \\\"config.json\\\").write_text(json.dumps(cfg, indent=2, default=str))\\n    report.update(\\n        {\\n            \\\"run_id\\\": run_id,\\n            \\\"features\\\": features_path,\\n            \\\"n_features\\\": len(cols),\\n            \\\"git\\\": git_hash(),\\n            \\\"seconds\\\": time.time() - t0,\\n        }\\n    )\\n    (out / \\\"metrics.json\\\").write_text(json.dumps(report, indent=2, default=float))\\n    append_jsonl(\\n        Path(cfg[\\\"paths\\\"][\\\"runs\\\"]).parent / \\\"experiments.jsonl\\\",\\n        {\\n            \\\"run_id\\\": run_id,\\n            \\\"final\\\": report[\\\"final\\\"][0],\\n            \\\"per_target\\\": report[\\\"final\\\"][1],\\n            \\\"stages\\\": {k: v[0] for k, v in report[\\\"stages\\\"].items()},\\n            \\\"seconds\\\": report[\\\"seconds\\\"],\\n        },\\n    )\\n    return report\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", default=\\\"configs/base.yaml\\\")\\n    ap.add_argument(\\\"--features\\\", default=\\\"data/processed/features_true.parquet\\\")\\n    ap.add_argument(\\\"--run_id\\\", default=None)\\n    ap.add_argument(\\\"--seed\\\", type=int, default=None)\\n    a = ap.parse_args()\\n    over = {\\\"seed\\\": a.seed} if a.seed is not None else None\\n    cfg = load_config(a.config, over)\\n    rid = a.run_id or time.strftime(\\\"%Y%m%d_%H%M%S\\\")\\n    rep = run(cfg, a.features, rid)\\n    print(\\n        json.dumps(\\n            {\\\"final\\\": rep[\\\"final\\\"], \\\"stages\\\": {k: v[0] for k, v in rep[\\\"stages\\\"].items()}},\\n            indent=2,\\n            default=float,\\n        )\\n    )\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"src/chume/validate.py\": \"from __future__ import annotations\\n\\nimport argparse\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef validate_submission(sub: pd.DataFrame, test_ids: pd.Series, id_col: str, targets: list[str]) -> None:\\n    exp = [id_col] + targets\\n    assert list(sub.columns) == exp, f\\\"columns {list(sub.columns)} != {exp}\\\"\\n    assert len(sub) == len(test_ids), f\\\"rows {len(sub)} != {len(test_ids)}\\\"\\n    assert sub[id_col].map(type).eq(str).all(), \\\"sample_id must be str\\\"\\n    assert not sub[id_col].duplicated().any(), \\\"duplicate ids\\\"\\n    assert set(sub[id_col]) == set(test_ids.astype(str)), \\\"id mismatch with test\\\"\\n    vals = sub[targets].to_numpy(dtype=float)\\n    assert np.isfinite(vals).all(), \\\"non-finite predictions\\\"\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"submission\\\")\\n    ap.add_argument(\\\"--test\\\", default=\\\"data/raw/test.csv\\\")\\n    ap.add_argument(\\\"--id\\\", default=\\\"sample_id\\\")\\n    a = ap.parse_args()\\n    targets = [\\\"t_ch4\\\", \\\"t_no2\\\", \\\"t_co\\\", \\\"t_aai\\\", \\\"t_firms\\\"]\\n    sub = pd.read_csv(a.submission, dtype={a.id: str})\\n    test = pd.read_csv(a.test, dtype={a.id: str})\\n    validate_submission(sub, test[a.id], a.id, targets)\\n    print(\\\"submission OK\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"pyproject.toml\": \"[build-system]\\nrequires = [\\\"setuptools>=68\\\"]\\nbuild-backend = \\\"setuptools.build_meta\\\"\\n\\n[project]\\nname = \\\"chume\\\"\\nversion = \\\"1.0.0\\\"\\nrequires-python = \\\">=3.10\\\"\\ndependencies = [\\n  \\\"numpy>=1.26\\\", \\\"pandas>=2.1\\\", \\\"pyarrow>=14\\\", \\\"scikit-learn>=1.4\\\", \\\"scipy>=1.11\\\",\\n  \\\"lightgbm>=4.3\\\", \\\"catboost>=1.2\\\", \\\"pyyaml>=6\\\", \\\"joblib>=1.3\\\", \\\"matplotlib>=3.8\\\",\\n  \\\"tifffile>=2024.1\\\", \\\"pillow>=10\\\", \\\"tabulate>=0.9\\\", \\\"optuna>=3.5\\\",\\n]\\n\\n[project.optional-dependencies]\\ngeo = [\\\"rasterio>=1.3\\\"]\\nemb = [\\\"torch>=2.2\\\", \\\"torchvision>=0.17\\\", \\\"torchgeo>=0.6\\\", \\\"timm>=0.9\\\"]\\ndev = [\\\"pytest>=8\\\"]\\n\\n[tool.setuptools.packages.find]\\nwhere = [\\\"src\\\"]\\n\", \"configs/base.yaml\": \"seed: 42\\npaths:\\n  raw: data/raw\\n  interim: data/interim\\n  processed: data/processed\\n  runs: outputs/runs\\n  train_csv: data/raw/train.csv\\n  test_csv: data/raw/test.csv\\n  chip_dir: data/raw/images\\n  sample_submission: data/raw/sample_submission.csv\\n\\ncolumns:\\n  id: sample_id\\n  site: site_key\\n  targets: [t_ch4, t_no2, t_co, t_aai, t_firms]\\n  wind_u: null\\n  wind_v: null\\n  wind_speed: wind_speed\\n  wind_dir: wind_direction\\n  wind_dir_convention: meteorological_from\\n  drop: []\\n\\nchips:\\n  pattern: \\\"{id}.tif\\\"\\n  band_order: [B01, B02, B03, B04, B05, B06, B07, B08, B8A, B09, B11, B12, SCL, CLDPRB]\\n  scale: 10000.0\\n  north_up: true\\n  scl_bad: [0, 1, 3, 8, 9, 10]\\n  cldprb_bad: 40\\n\\nfeatures:\\n  generic_chip: true\\n  plume: true\\n  plume_random_angle: false\\n  plume_radii: [0.10, 0.25, 0.45]\\n  plume_half_angle_deg: 30\\n  calm_ws: 1.0\\n  massbalance: true\\n  embeddings: false\\n  embeddings_model: ResNet50_Weights.SENTINEL2_ALL_MOCO\\n  embeddings_pca: 16\\n  site_context: false\\n  drop_regex: []\\n  keep_regex: []\\n\\npost:\\n  clip_train_range: false\\n\\ncv:\\n  n_folds: 5\\n  seeds: [42, 7, 2026]\\n\\ntargets:\\n  transform: auto\\n  candidates: [none, ws_mult, slog]\\n\\nmodels:\\n  lgbm: true\\n  catboost: true\\n  n_model_seeds: 3\\n  chain: true\\n  chain_include_self: false\\n  gate: true\\n  gate_w_max: 1.2\\n  blend: true\\n\\nlgbm_params:\\n  objective: regression\\n  learning_rate: 0.03\\n  num_leaves: 31\\n  min_data_in_leaf: 40\\n  feature_fraction: 0.7\\n  bagging_fraction: 0.8\\n  bagging_freq: 1\\n  lambda_l1: 0.1\\n  lambda_l2: 1.0\\n  n_estimators: 3000\\n  early_stopping_rounds: 150\\n  verbose: -1\\n\\ncatboost_params:\\n  loss_function: RMSE\\n  learning_rate: 0.03\\n  depth: 6\\n  l2_leaf_reg: 5.0\\n  iterations: 3000\\n  od_wait: 150\\n  verbose: 0\\n  allow_writing_files: false\\n\", \"configs/final.json\": \"{\\n \\\"features\\\": {\\n  \\\"plume\\\": true,\\n  \\\"massbalance\\\": true,\\n  \\\"generic_chip\\\": true,\\n  \\\"drop_regex\\\": [\\n   \\\"^(p_|g_|scl_|hot_|tai_|mb_|clear)\\\"\\n  ],\\n  \\\"keep_regex\\\": [\\n   \\\"^(tai_n_pos_all|tai_max|hot_n2\\\\\\\\.0_all|hot_n1\\\\\\\\.5_all|hot_ratio_max|hot_n2\\\\\\\\.0_core|mb_ws|mb_u|mb_v|mb_inv_ws|mb_log_ws|mb_tai_max_per_ws|mb_tai_n_pos_all_per_ws|g_B(0[1-8]|8A|09|11|12)_p50|scl_(4|5|6|10)|p_haze_core_p90|p_r_ch4_norm_du0\\\\\\\\.25|p_r_ch4_norm_duz0\\\\\\\\.25|p_r_ch4_norm_cr0\\\\\\\\.25|p_tai_core_max|p_hot_ratio_core_max|p_haze_du0\\\\\\\\.25|p_b12_du0\\\\\\\\.25|mb_p_r_ch4_norm_du0\\\\\\\\.25_x_ws)$\\\"\\n  ]\\n },\\n \\\"columns\\\": {\\n  \\\"drop\\\": [\\n   \\\"elevation\\\",\\n   \\\"slope\\\"\\n  ]\\n },\\n \\\"lgbm_params\\\": {\\n  \\\"num_leaves\\\": 7,\\n  \\\"min_data_in_leaf\\\": 80,\\n  \\\"feature_fraction\\\": 0.5,\\n  \\\"learning_rate\\\": 0.02\\n },\\n \\\"models\\\": {\\n  \\\"catboost\\\": false,\\n  \\\"chain\\\": true,\\n  \\\"blend\\\": true,\\n  \\\"gate\\\": false\\n },\\n \\\"targets\\\": {\\n  \\\"transform\\\": \\\"auto\\\"\\n }\\n}\"}")
root = pathlib.Path('/kaggle/working/chume')
for rel, txt in FILES.items():
    p = root / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt)
print(len(FILES), 'files written')

In [ ]:
%cd /kaggle/working/chume
!pip install -q --no-deps --no-build-isolation -e .
import glob
IN = sorted(glob.glob('/kaggle/input/*/train.csv'))[0].rsplit('/', 1)[0]
print('input dir', IN)

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/base.yaml'))
cfg['paths'].update({'raw': IN, 'train_csv': f'{IN}/train.csv', 'test_csv': f'{IN}/test.csv',
    'chip_dir': f'{IN}/images', 'sample_submission': f'{IN}/sample_submission.csv',
    'processed': '/kaggle/working/processed', 'runs': '/kaggle/working/runs'})
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'))
!python -m chume.features.build --config configs/kaggle.yaml --n_jobs 4

In [ ]:
import json
import numpy as np, pandas as pd
from chume.config import load_config
from chume.train import run
over = json.load(open('configs/final.json'))
subs, cvs = [], []
for seed in (42, 7, 2026):  # the 3 CV seeds; test predictions are averaged
    rep = run(load_config('configs/kaggle.yaml', {**over, 'seed': seed}),
              '/kaggle/working/processed/features_true.parquet', f'final_s{seed}')
    cvs.append(rep['final'][0])
    subs.append(pd.read_csv(f'/kaggle/working/runs/final_s{seed}/submission.csv', dtype={'sample_id': str}))
    print(seed, 'CV MCRMSE', round(rep['final'][0], 5), {k: round(v, 4) for k, v in rep['final'][1].items()})
print('mean CV', np.mean(cvs), '+-', np.std(cvs))
T = ['t_ch4', 't_no2', 't_co', 't_aai', 't_firms']
sub = subs[0].copy()
sub[T] = np.mean([s[T].values for s in subs], axis=0)

In [ ]:
import pandas as pd, shutil
from chume.validate import validate_submission
test = pd.read_csv(f'{IN}/test.csv', dtype={'sample_id': str})
validate_submission(sub, test['sample_id'], 'sample_id',
                    ['t_ch4', 't_no2', 't_co', 't_aai', 't_firms'])
sub.to_csv('/kaggle/working/submission.csv', index=False)
sub.head()